In [ ]:
!pip install -q sacremoses sentencepiece

In [ ]:
import torch
from transformers import AutoTokenizer, AutomodelForSeq2SeqLM
from tqdm import tqdm

In [ ]:
input_file = "en_pairs.txt"
output_file = "en_translated_pairs.txt"

device = "cuda"

model = "facebook/nllb-200-distilled-600M"
batch_size = 32  

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutomodelForSeq2SeqLM.from_pretrained(
    model,
    torch_dtype=torch.float16
).to(device)

model.eval()

tokenizer.src_lang = "eng_Latn"
forced_bos_token_id = tokenizer.convert_tokens_to_ids("nld_Latn")

In [ ]:
lines = []

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n")
        parts = [p.strip() for p in line.split("||")]

        english = parts[1]
        rest = [parts[0]] + parts[2:]

        lines.append((english, rest))

In [ ]:
# deduplicate
unique_terms = list(dict.fromkeys(english for english, _ in lines if english))

translations = {}


for i in tqdm(range(0, len(unique_terms), batch_size)):
    batch = unique_terms[i:i + batch_size]

    encoded = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        generated = model.generate(
            **encoded,
            forced_bos_token_id=forced_bos_token_id,
            num_beams=8,
            max_length=256
        )

    decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)

    for src, tgt in zip(batch, decoded):
        translations[src] = tgt

In [ ]:
out_lines = []

for english, rest in lines:
    dutch = translations.get(english, english)
    cui = rest[0]
    tail = rest[1:]
    out_lines.append("||".join([cui, dutch] + tail))

with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(out_lines))